# Home Credit Default Risk

Walkthrough оценки платёжных трудностей по анкете и истории клиента.
Учебный путь по умолчанию выполняет небольшой синтетический smoke-run.
Его метрики не оценивают качество Home Credit.

В конце notebook показан отдельный **проверенный реальный эксперимент** из
`reports/home-credit-full-001.json`. Это агрегированный публичный отчёт:
исходные CSV, прогнозы клиентов, модели и split indices в Git не входят.
Подробности: [README](README.md), [результаты](RESULTS.md),
[методика](docs/methodology.md), [данные](DATA_ACCESS.md).

## 1. Фиксируем эксперимент до просмотра метрик

Сначала резервируем независимый финальный holdout (по умолчанию 20%). Он
не участвует ни в одном outer fold, подборе модели, отборе признаков или порога.
На остальных development-данных выполняется nested CV: outer train делится
на fit / tune / calibration. После CV та же заранее заданная процедура обучается
на development, фиксирует прогнозы и только затем оценивается на holdout.

Три ветки с одним LightGBM: `raw`, `application`, `history`. Разница
`history − application` относится к добавлению истории в рамках этой процедуры.
Основная политика `selected` выбирается по внутреннему tune AUC. План разбиения
и hashes кода/входов сохраняются до обучения. После просмотра финального holdout
он перестаёт быть независимым для последующих изменений модели.

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import json
from dataclasses import replace

import pandas as pd
from credit_scoring.protocol import Config
from credit_scoring.features import check_files, build_features
from credit_scoring.independent import independent_evaluate
from credit_scoring.run import input_manifest, save_evaluation, environment, write_plan, verify_inputs
from credit_scoring.demo import synthetic_data

DATA = Path("home-credit-default-risk")
RUN_REAL = False       # True только после получения всех CSV
RUN_SUBMISSION = False # отдельный явный запуск; файл никуда не отправляется
FAMILIES = ("lightgbm",)  # optional: ("lightgbm", "xgboost", "catboost")
CONFIG = (Config(outer_folds=3, max_rounds=600, patience=50, probe_rounds=100,
                 threads=4, families=FAMILIES) if RUN_REAL else
          Config(outer_folds=2, inner_folds=3, max_rounds=40, patience=8,
                 probe_rounds=15, top_k=12, complexities=(7, 15), threads=1,
                 families=FAMILIES))
MODE = "home-credit" if RUN_REAL else "synthetic"
print("Режим:", MODE)
print(json.dumps(CONFIG.as_dict(), indent=2))
print(json.dumps(environment(), indent=2))

## 2. Проверка данных и структуры

Нужны восемь CSV, перечисленные в README. Данные не скачиваются автоматически.
В реальном режиме загрузчик останавливается при дубликатах application IDs,
пересечении train/test или противоречивых владельцах прошлых кредитов.
Будущие исторические наблюдения исключаются. Агрегация по владельцу
детерминирована и не использует TARGET; vocabulary, фильтры и supervised отбор
обучаются позже внутри training partitions.

SK_ID_CURR не позволяет доказать уникальность физического лица между разными ID.
Абсолютных дат текущих заявок в ожидаемой схеме нет: default CV предполагает IID,
а не перенос во времени. Для подтверждённых borrower groups / timestamps
используйте group/time API из README на всех уровнях. В исходных application CSV абсолютная дата заявки не обнаружена;
forward-time оценка по ним недоступна.

Платежи сначала сворачиваются до взноса по SK_ID_PREV, версии, номеру и дате планового платежа (`DAYS_INSTALMENT`).
Частичные платежи суммируются, дата погашения — момент покрытия суммы взноса;
неоплаченные взносы сохраняются с просрочкой на дату заявки. Платежи из будущего
не становятся наблюдавшимися деньгами. Для объединения между CSV-порциями
используется временная SQLite-база на локальном диске, удаляемая после обработки.

`AMT_ANNUITY / AMT_CREDIT` — отношение аннуитетного платежа к сумме кредита
(`ANNUITY_CREDIT_RATIO`). Формула признака сохранена; отношение не определяет
фактическую длительность кредитного договора.

In [ ]:
try:
    check_files(DATA)
    print("Все восемь файлов присутствуют; содержимое проверяется при загрузке.")
except FileNotFoundError as error:
    print(error)
    if RUN_REAL:
        raise
    print("Продолжаем только явно выбранный synthetic smoke-run.")

if RUN_REAL:
    manifest = input_manifest(DATA)
    frames, y, test_frames, test_ids, data_audit = build_features(DATA)
else:
    frames, y = synthetic_data(CONFIG.seed)
    manifest = {}
    test_frames, test_ids = None, None
    data_audit = {"synthetic": True, "rows": len(y),
                  "warning": "Artificial smoke data, not a Home Credit performance estimate"}
print(data_audit)
pd.DataFrame({name: {"rows": len(x), "features_before_fitted_transform": x.shape[1]}
              for name, x in frames.items()}).T

## 3. Запуск фиксированной процедуры

Для реального режима полный runtime и расход памяти пока не измерены. Две
сложности и предел числа итераций ограничивают поиск; все потоки заданы явно.
В синтетическом режиме ниже действительно выполняется обучение, но оно
не подтверждает бизнес-качество или величину улучшения на Home Credit.

In [ ]:
run_id = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
OUTPUT = Path("artifacts") / f"notebook-{MODE}-{run_id}"
if RUN_REAL:
    verify_inputs(DATA, manifest)
partitions = write_plan(OUTPUT, frames, y, CONFIG, data_audit, manifest)
experiment = independent_evaluate(frames, y, CONFIG, partitions=partitions)
result = experiment["development"]
print("Nested CV только на development, режим:", MODE)
result["summary"].round(5)

Средняя AUC считается по development outer folds; std — разброс folds,
не доверительный интервал. Стоимость, Brier и log loss взвешиваются по числу
заявок. `selected` оценивает внутреннюю процедуру выбора, модель которой
может различаться между folds. Эти метрики не включают финальный holdout.
Повторный выбор по outer-таблице создаст новый цикл настройки.

In [ ]:
print("Парные разницы AUC, режим:", MODE)
result["ablation"].round(5)

### Независимый финальный holdout

Все прогнозы и пороги были зафиксированы до вычисления этой таблицы.
Основная строка — заранее объявленная `selected`; остальные сравнения
описательные и не используются для выбора новой модели. Доверительные
интервалы пока не рассчитываются; преимущество по одной точечной оценке
не доказывает устойчивого улучшения.

In [ ]:
print("Независимый holdout, режим:", MODE)
experiment["holdout_metrics"].round(5)

## 4. Порог и стоимость

Учебный сценарий: пропущенный дефолт стоит 10, отказ хорошему клиенту — 1.
Это допущение, а не оценённая экономика портфеля. Отказываем при `p >= threshold`.
Порог выбирается **только на calibration**, одинаковые скоры движутся вместе.
Варианты отказать всем и одобрить всем также проверяются.
Ниже стоимость на outer fold при уже фиксированном пороге; она не оптимизируется
повторно по outer-меткам. При отсутствии одобрений bad rate не определён.

Метки TARGET соответствуют payment difficulties в постановке соревнования;
цены ошибок 10:1 не являются оценкой фактического денежного ущерба.

In [ ]:
cols = ["fold", "representation", "family", "threshold", "auc", "cost",
        "approval_rate", "approved_default_rate", "approve_all_cost", "reject_all_cost"]
result["folds"].query("policy == 'selected'")[cols]

## 5. Аудит и сохранение

В report сохраняются настройки, версии, SHA-256 входных файлов и исходников,
строки всех partitions, выбранные признаки и итерации. `predictions.csv` содержит
метки и не должен публиковаться. Каждый запуск создаёт отдельный каталог.
Результаты синтетического режима явно помечены.

`calibration` здесь означает выбор порога, а не Platt/isotonic-калибровку
вероятностей. Для диагностики добавлены Brier score и log loss. Сохраняется
готовая модель с преобразованиями, а не только число итераций.

In [ ]:
held_rows = set(experiment["holdout_audit"]["holdout_rows"])
for fold in result["audit"]:
    outer = set(fold["outer_validation"])
    assert not held_rows.intersection(fold["outer_train"])
    assert not held_rows.intersection(outer)
    for policy in fold["policies"].values():
        assert not outer.intersection(policy["refit_rows"])
        assert not outer.intersection(policy["calibration_rows"])
        assert not set(policy["refit_rows"]).intersection(policy["calibration_rows"])
for policy in experiment["policies"].values():
    assert not held_rows.intersection(policy.metadata["refit_rows"])
    assert not held_rows.intersection(policy.metadata["calibration_rows"])
print("Проверена изоляция независимого holdout и всех внутренних partitions.")
if RUN_REAL:
    verify_inputs(DATA, manifest)
save_evaluation(result, OUTPUT, CONFIG, data_audit, manifest, independent=experiment)
print("Локальные файлы:", OUTPUT)
print(sorted(p.name for p in OUTPUT.iterdir()))

## 6. Необязательный финальный прогноз

Для реального submission используем **ту же development-модель**, которая
оценивалась на независимом holdout. Она уже настроена и сохранена вместе с
преобразованиями в `final_policy.joblib` и metadata в `final_policy.json`.
Нового обучения, в том числе на holdout, здесь нет. Прогнозы сохраняются
локально в исходном порядке test IDs. Бизнес-порог не превращает TARGET в
доказанный дефолт: это учебный сценарий на метке соревнования.

In [ ]:
if RUN_SUBMISSION:
    if not RUN_REAL:
        raise ValueError("Submission доступен только для реальных данных")
    final_policy = experiment["policies"]["selected"]
    submission = pd.DataFrame({"SK_ID_CURR": test_ids,
                               "TARGET": final_policy.predict(test_frames)})
    submission.to_csv(OUTPUT / "submission.csv", index=False)
    print("Сохранён локальный submission.csv:", len(submission), "строк")
else:
    print("Финальный submission не запрошен.")

## Что можно заключить после запуска

В synthetic-режиме — только то, что код и проверки выполняются. Даже отдельный
holdout синтетического примера не оценивает качество Home Credit.
Реальные выводы о AUC, стоимости и вкладе истории требуют исходных CSV и
заранее зафиксированного эксперимента. Их нельзя переносить из старого notebook.

После просмотра финального holdout его нельзя использовать как независимое
подтверждение новых правок. Для временного переноса также нужны настоящие даты
заявок; static random split его не доказывает. Полные ограничения и инструкции
находятся в README, а описание датасета — в DATA_ACCESS.md.

## Зафиксированный полный эксперимент на Home Credit

Читаем публичные агрегаты `reports/home-credit-full-001.json` без повторного
обучения. Основная политика `selected` выбрана на внутренних данных до final holdout.
Development CV и независимый holdout показаны отдельно. Парные bootstrap-интервалы
условны на фиксированных моделях и числах классов при IID-предположении;
они не покрывают перенос во времени или неопределённость переобучения.
Полный разбор находится в RESULTS.md.

In [9]:
from IPython.display import display
PUBLIC_REPORT = Path("reports/home-credit-full-001.json")
recorded = json.loads(PUBLIC_REPORT.read_text())
print("РЕАЛЬНЫЕ ДАННЫЕ — development nested CV")
display(pd.DataFrame(recorded["development_summary"]).set_index("policy").round(6))
print("РЕАЛЬНЫЕ ДАННЫЕ — независимый финальный holdout")
display(pd.DataFrame(recorded["holdout_metrics"]).set_index("policy").round(6))
print("Парный стратифицированный bootstrap: условные 95% интервалы AUC")
display(pd.DataFrame(recorded["auc_intervals"]).round(6))
display(pd.DataFrame(recorded["paired_auc_differences"]).round(6))

РЕАЛЬНЫЕ ДАННЫЕ — development nested CV


,mean_fold_auc,std_fold_auc,mean_fold_ap,evaluated_rows,cost,approval_rate,approve_all_cost,reject_all_cost,brier,log_loss
policy,,,,,,,,,,
application_lightgbm,0.761618,0.003176,0.245828,246009,0.525875,0.711222,0.807288,0.919271,0.067681,0.245296
history_lightgbm,0.785386,0.004032,0.276432,246009,0.491730,0.715409,0.807288,0.919271,0.066095,0.237933
raw_lightgbm,0.754328,0.004482,0.238907,246009,0.535810,0.723555,0.807288,0.919271,0.068034,0.247272
selected,0.785386,0.004032,0.276432,246009,0.491730,0.715409,0.807288,0.919271,0.066095,0.237933


РЕАЛЬНЫЕ ДАННЫЕ — независимый финальный holdout


,representation,family,threshold,cost,fn,fp,approval_rate,approved_default_rate,approve_all_cost,reject_all_cost,auc,ap,brier,log_loss,n
policy,,,,,,,,,,,,,,,
raw_lightgbm,raw,lightgbm,0.089117,0.536714,1821,14799,0.708253,0.041805,0.807291,0.919271,0.753314,0.236246,0.068141,0.247582,61502
application_lightgbm,application,lightgbm,0.090031,0.525609,1780,14526,0.712026,0.040648,0.807291,0.919271,0.761651,0.246585,0.067656,0.245181,61502
history_lightgbm,history,lightgbm,0.082938,0.485675,1491,14960,0.700270,0.034620,0.807291,0.919271,0.787863,0.288700,0.065626,0.236443,61502
selected,history,lightgbm,0.082938,0.485675,1491,14960,0.700270,0.034620,0.807291,0.919271,0.787863,0.288700,0.065626,0.236443,61502


Парный стратифицированный bootstrap: условные 95% интервалы AUC


,policy,auc,lower_95,upper_95
0,application_lightgbm,0.761651,0.754833,0.768448
1,history_lightgbm,0.787863,0.781843,0.794796
2,raw_lightgbm,0.753314,0.746473,0.760159
3,selected,0.787863,0.781843,0.794796


,comparison,delta_auc,lower_95,upper_95
0,history_lightgbm_minus_application_lightgbm,0.026211,0.022616,0.029782
1,application_lightgbm_minus_raw_lightgbm,0.008337,0.005972,0.010795
2,history_lightgbm_minus_raw_lightgbm,0.034549,0.030281,0.038721
